In [6]:
"""
qwen2.5vl:3b
"""
import ollama
import base64
import json

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def chat_once(user_text, image_path=None, history=None):
    if history is None:
        history = []
    
    if image_path:
        message = {
            "role": "user",
            "content": user_text,
            "images": [encode_image(image_path)]
        }
    else:
        message = {
            "role": "user",
            "content": user_text
        }
    
    history.append(message)
    
    response = ollama.chat(
        model="qwen2.5vl:3b",
        messages=history
    )
    
    return response["message"]["content"]


image_path = "Scene/Scene1.png"
target_question = "what is to the left of the sofa and what is the colour of it?"

conditions = {
    "relative": f"From your perspective looking at the scene, {target_question}",
    "intrinsic": f"From the sofa's own perspective, {target_question}",
    "hearer_180": f"I am standing opposite you. From my perspective, {target_question}"
}

results = {}

for condition_name, prompt in conditions.items():
    answer = chat_once(prompt, image_path=image_path, history=[])
    results[condition_name] = answer
    print(f"[{condition_name}]")
    print(answer)
    print()

with open("task1_results_qwen.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

[relative]
To the left of the sofa, there is a wooden chair. The chair is brown in color.

[intrinsic]
From the sofa's own perspective, to the left of the sofa is a wooden chair. The chair is brown in color.

[hearer_180]
From your perspective, to the left of the sofa is a wooden chair. The chair is brown in color.



In [ ]:
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

conversation_history = []

def chat(user_text, image_path=None):
    if image_path:
        message = {
            "role": "user",
            "content": user_text,
            "images": [encode_image(image_path)]
        }
    else:
        message = {
            "role": "user", 
            "content": user_text
        }
    
    conversation_history.append(message)
    
    response = ollama.chat(
        model="qwen2.5vl:3b",
        messages=conversation_history
    )
    
    assistant_message = {
        "role": "assistant",
        "content": response["message"]["content"]
    }
    conversation_history.append(assistant_message)
    
    return response["message"]["content"]


image_path = "Scene/Scene1.png"

turn1 = chat("There is a TV in front of the sofa. Can you see it?", image_path=image_path)
print("T1:", turn1)

turn2 = chat("There is a chair to the left of the sofa, which chair is it and what's the colour of the chair?")
print("T2:", turn2)

turn3 = chat("There is a table to the right of the sofa, which table is it and what's the colour of the chair?")
print("T3:", turn3)

turn4 = chat("Where is the wardrobe? Please describe its location.")
print("T4:", turn4)

results = {
    "scene": image_path,
    "turn1_priming": turn1,
    "turn2_probe": turn2,
    "turn3_probe": turn3,
    "turn4_free_description": turn4
}

with open("task2_results_qwen.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

T1: The image depicts a simple room with a few pieces of furniture. Here is a detailed description:

1. **Sofa**: There is a gray sofa in the center of the room. It has two cushions and is positioned against the wall.
2. **Chair**: To the left of the sofa, there is a wooden chair.
3. **Table**: In front of the sofa, there is a wooden table with a white top and wooden legs.
4. **Lamps**: There are two floor lamps in the room. One is on the left side, and the other is on the right side.
5. **Cabinet**: On the left side of the room, there is a wooden cabinet with glass doors.
6. **Person**: A person is standing near the center of the room, slightly to the right of the sofa.
7. **Television**: There is a television mounted on the wall to the right of the sofa.

The room has a simple and clean design, with a neutral color palette.
T2: The chair to the left of the sofa is wooden.
T3: The table to the right of the sofa is wooden.
T4: The wardrobe is located on the left side of the room. It is

In [ ]:
"""
Gemini-3.1-flash-lite
"""
from google import genai
from PIL import Image
import json

client = genai.Client(api_key)

chat = client.chats.create(model="gemini-3.1-flash-lite")

image_path = "Scene/Scene1.png"
image = Image.open(image_path)

target_question = "what is to the left of the sofa and what is the colour of it?"

conditions = {
    "relative": f"From your perspective looking at the scene, {target_question}",
    "intrinsic": f"From the sofa's own perspective, {target_question}",
    "hearer_180": f"I am standing opposite you. From my perspective, {target_question}"
}

results = {}

for condition_name, prompt in conditions.items():
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=[prompt, image]
    )
    results[condition_name] = response.text
    print(f"[{condition_name}]")
    print(response.text)
    print()

with open("task1_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

[relative]
To the left of the sofa, there is a brown wooden chair.

[intrinsic]
From the perspective of someone sitting on the sofa and facing forward, to the left of the sofa is a brown wooden **chair**.

[hearer_180]
If you are standing opposite me (facing the sofa from the side where the person is standing), "left" from your perspective would be the side where the table and television are located.

Specifically, to the left of the sofa from your perspective is a **white table**, followed by the black television further to the left.



In [3]:
response1 = chat.send_message([
    "There is a TV in front of the sofa.",
    image
])
print("T1:", response1.text)

response2 = chat.send_message(
    "There is a chair to the left of the sofa, which chair is it and what's the colour of the chair?"
)
print("T2:", response2.text)

response3 = chat.send_message(
    "There is a table to the right of the sofa, which table is it and what's the colour of the table?"
)
print("T3:", response3.text)

response4 = chat.send_message(
    "Where is the wardrobe? Please describe its location."
)
print("T4:", response4.text)

results = {
    "scene": image_path,
    "turn1_priming": response1.text,
    "turn2_probe": response2.text,
    "turn3_probe": response3.text,
    "turn4_free_description": response4.text
}

with open("task2_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

T1: I apologize for the confusion. Looking at the image again from the perspective of the sofa, the TV is positioned to its right. 

If you are referring to the perspective of the person standing behind the sofa, the TV is indeed situated on the right side of the room, rather than directly in front of the seating area. There is no furniture positioned directly in front of the sofa except for the open floor space and the large wooden table further in the foreground.
T2: As mentioned previously, there is a brown wooden dining-style chair located to the left of the sofa.
T3: The table to the right of the sofa is a dining table. It has a white rectangular top and light wooden legs.
T4: The wardrobe is situated on the far left side of the room. It is a tall, wooden piece of furniture positioned behind the brown chair, standing against the back left area of the scene. It features a light brown wood finish with a grey cabinet section and a set of drawers stacked below it.
